# Bayesian (BF10) follow-up for the partially-supported hypotheses
**فقط `Runtime → Run all` بزن.** هیچ Mapper یا embedding ای دوباره اجرا نمی‌شود؛ فقط از CSV های نتایج قبلی استفاده می‌کند (چند ثانیه تا ۱ دقیقه).
وقتی پنجره‌ی آپلود باز شد، فایل **`final_results.zip`** (همان خروجی اجرای نهایی، نه zip مربوط به baseline) را انتخاب کن.

**این تحلیل post hoc است** و حکم‌های قبلی (قاعده‌های از پیش‌تعیین‌شده) را تغییر نمی‌دهد؛ فقط «شدت شواهد» را کنارشان می‌گذارد. قواعد تفسیر BF و واحد تحلیل (seed های مستقل) پیش از دیدن نتایج ثابت شده‌اند.

In [ ]:
import os, io, zipfile, glob, warnings
import numpy as np, pandas as pd
from scipy import stats as st
from scipy.special import logsumexp
warnings.filterwarnings("ignore")

# ---------- بارگذاری فایل‌ها ----------
DATA_DIR = None
try:
    from google.colab import files
    print("فایل final_results.zip را آپلود کن:")
    up = files.upload()
    fn = [k for k in up if k.lower().endswith(".zip")][0]
    os.makedirs("fr_in", exist_ok=True)
    zipfile.ZipFile(io.BytesIO(up[fn])).extractall("fr_in"); DATA_DIR = "fr_in"
except ImportError:
    DATA_DIR = os.environ.get("BF_DATA_DIR", ".")
need = ["table3_tier_design_by_seed_primary.csv", "table8_real_disagreement_socialchem.csv", "table9_real_disagreement_scruples.csv", "table2_main_graphs_formal_tests.csv"]
def find(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    assert hits, f"{name} در zip پیدا نشد. zip درست (اجرای نهایی) را آپلود کن."
    return hits[0]
T3 = pd.read_csv(find(need[0])); T8 = pd.read_csv(find(need[1])); T9 = pd.read_csv(find(need[2])); T2 = pd.read_csv(find(need[3]))
print({k: v.shape for k, v in dict(T3=T3, T8=T8, T9=T9, T2=T2).items()})

## قواعد ثابت (قبل از اجرا)

In [ ]:
# ===== قواعد از پیش تعیین‌شده =====
# 1) آزمون: JZS Bayesian one-sample/paired t-test (prior: Cauchy روی اندازه‌ی اثر استاندارد). پیش‌فرض r = 0.707؛ حساسیت به r در {0.5, 0.707, 1.0}.
# 2) واحد تحلیل = seed های مستقل نمونه‌گیری (n = 5) روی میانگین lens ها. گراف/lens/encoder به‌عنوان تکرار مستقل شمرده نمی‌شوند (pseudo-replication).
# 3) آزمون جهت‌دار: BF+0 برای فرضیه‌ی مورد انتظار (اثر > 0). برای H9 فرضیه‌ی اصلی جهت منفی است (pressure < judgment) و جهت معکوس هم گزارش می‌شود.
# 4) تفسیر (Lee & Wagenmakers): BF>=10 قوی؛ 3..10 متوسط؛ 1/3..3 نامطمئن؛ 1/10..1/3 شاهد متوسط برای H0؛ <=1/10 شاهد قوی برای H0.
# 5) دسته‌ی نهایی با «محافظه‌کارانه‌ترین» r (کمترین BF در سه r) تعیین می‌شود؛ اگر دسته‌ها بین r ها فرق کنند «prior-sensitive» علامت می‌خورد.
# 6) همه‌ی آزمون‌ها، از جمله آن‌هایی که به نفع H0 هستند، گزارش می‌شوند.
INFORMATIVE = ["pca_1d", "centroid_1d", "eccentricity_1d", "density_1d"]
R_LIST = [0.5, 0.707, 1.0]; R_MAIN = 0.707

def jzs_bf(t, n, r=0.707, ngrid=6001):
    df = n - 1; d_hat = t / np.sqrt(n); span = max(8.0, 2.0 * abs(d_hat) + 8.0)
    grid = np.linspace(-span, span, ngrid); step = grid[1] - grid[0]
    ll = st.nct.logpdf(t, df, grid * np.sqrt(n)); ll0 = st.t.logpdf(t, df)
    ll = np.where(np.isfinite(ll), ll, -np.inf)
    lw = ll + st.cauchy.logpdf(grid, 0, r) + np.log(step) - ll0
    la = logsumexp(lw); lp = logsumexp(lw[grid > 0]); ln_ = logsumexp(lw[grid < 0]); ln10 = np.log(10)
    return la / ln10, (lp + np.log(2)) / ln10, (ln_ + np.log(2)) / ln10      # log10 BF10, BF+0, BF-0

def label(bf, bf_opp=None, claim="effect"):
    if bf >= 10: return f"STRONG for {claim}"
    if bf >= 3: return f"MODERATE for {claim}"
    if bf > 1/3: return "INCONCLUSIVE"
    if bf_opp is not None and bf_opp >= 3: return "EVIDENCE FOR OPPOSITE DIRECTION"
    if bf > 1/10: return "MODERATE for null"
    return "STRONG for null"

def fmt(x):
    return f">1e6" if x > 1e6 else (f"<1e-6" if x < 1e-6 else (f"{x:.1f}" if x >= 10 else f"{x:.2f}"))

ROWS = []
def test_vs_zero(hid, name, x, direction=+1):
    """x: seed-level values (n=5). direction=+1 tests effect>0 ; -1 tests effect<0."""
    x = np.asarray(pd.Series(x).dropna(), float); n = len(x)
    m, sd = x.mean(), x.std(ddof=1); t = (m / (sd / np.sqrt(n))) if sd > 1e-12 else np.sign(m) * 1e3
    out = dict(ID=hid, test=name, n_seeds=n, mean=m, sd=sd, dz=(m / sd if sd > 1e-12 else np.nan), t=t, direction=">0" if direction > 0 else "<0")
    cats, bfs = [], []
    for r in R_LIST:
        l10, lp, lm = jzs_bf(t, n, r)
        bf_dir = 10 ** (lp if direction > 0 else lm); bf_opp = 10 ** (lm if direction > 0 else lp)
        bfs.append(bf_dir); cats.append(label(bf_dir, bf_opp))
        if r == R_MAIN: out["BF10_two_sided"] = 10 ** l10
        out[f"BF_dir_r{r}"] = bf_dir
    out["BF_dir_main"] = out[f"BF_dir_r{R_MAIN}"]; out["BF_dir_min"] = min(bfs)
    i = int(np.argmin(bfs)); out["category"] = cats[i]; out["prior_sensitive"] = len(set(cats)) > 1
    ROWS.append(out); return out
print("rules ready")

## تحلیل‌ها (همه روی seed های مستقل)

In [ ]:
# ---------- A) H1 / H2 : خلوص و توافق یال‌ها نسبت به شانس، مدل اصلی، ۵ seed ----------
a = (T3[T3.lens.isin(INFORMATIVE)].groupby(["seed", "task", "tier"])[["CP_delta", "EA_delta"]].mean().reset_index())
for task in ["judgment", "pressure", "foundation"]:
    for tier in ["High", "Mid", "Low"]:
        s = a[(a.task == task) & (a.tier == tier)].sort_values("seed")
        test_vs_zero("H1", f"dCP>0 | {task} | {tier}", s.CP_delta); test_vs_zero("H2", f"dEA>0 | {task} | {tier}", s.EA_delta)
for tier in ["High", "Mid", "Low"]:                         # میانگین سه تسک در هر seed
    s = a[a.tier == tier].groupby("seed")[["CP_delta", "EA_delta"]].mean()
    test_vs_zero("H1", f"dCP>0 | ALL TASKS | {tier}", s.CP_delta); test_vs_zero("H2", f"dEA>0 | ALL TASKS | {tier}", s.EA_delta)

# ---------- B) H7d : High - Low (paired seeds) ----------
for task in ["judgment", "pressure", "foundation"]:
    pv = a[a.task == task].pivot(index="seed", columns="tier", values=["CP_delta", "EA_delta"])
    test_vs_zero("H7d", f"dCP High-Low>0 | {task}", pv["CP_delta"]["High"] - pv["CP_delta"]["Low"])
    test_vs_zero("H7d", f"dEA High-Low>0 | {task}", pv["EA_delta"]["High"] - pv["EA_delta"]["Low"])

# ---------- C) H9 : pressure vs judgment (High+Mid) ----------
hm = a[a.tier.isin(["High", "Mid"])].groupby(["seed", "task"])[["CP_delta", "EA_delta"]].mean().reset_index()
for met, nm in [("CP_delta", "dCP"), ("EA_delta", "dEA")]:
    pv = hm.pivot(index="seed", columns="task", values=met); d = pv["pressure"] - pv["judgment"]
    test_vs_zero("H9", f"{nm} pressure-judgment<0 (original H9: pressure MORE interwoven)", d, direction=-1)
    test_vs_zero("H9x", f"{nm} pressure-judgment>0 (exploratory reverse)", d, direction=+1)

# ---------- D) H7a / H7b / H7c : اختلاف واقعی annotator ها ----------
real = pd.concat([T8, T9], ignore_index=True)
ps = real.groupby(["dataset", "model", "group", "seed"])[["dCP", "dEA", "rho"]].mean().reset_index()
def contrast(df, met):
    pv = df.pivot(index="seed", columns="group", values=met); return pv["Unanimous"] - pv["Strong"]
for ds, g in ps.groupby("dataset"):
    for mo, gm in g.groupby("model"):
        test_vs_zero("H7a", f"dCP Unan-Strong>0 | {ds} | {mo}", contrast(gm, "dCP"))
        test_vs_zero("H7b", f"dEA Unan-Strong>0 | {ds} | {mo}", contrast(gm, "dEA"))
        test_vs_zero("H7c", f"rho pooled>0 | {ds} | {mo}", gm[gm.group == "POOLED"].sort_values("seed").rho)
    gg = g.groupby(["group", "seed"])[["dCP", "dEA", "rho"]].mean().reset_index()      # میانگین سه encoder در هر seed
    test_vs_zero("H7a", f"dCP Unan-Strong>0 | {ds} | ALL ENCODERS", contrast(gg, "dCP"))
    test_vs_zero("H7b", f"dEA Unan-Strong>0 | {ds} | ALL ENCODERS", contrast(gg, "dEA"))
    test_vs_zero("H7c", f"rho pooled>0 | {ds} | ALL ENCODERS", gg[gg.group == "POOLED"].sort_values("seed").rho)

res = pd.DataFrame(ROWS); print(len(res), "tests")

## H4 — سازگاری بین encoder ها (همبستگی بیزی؛ توصیفی چون سلول‌ها مستقل نیستند)

In [ ]:
# تقریب Fisher-z با prior یکنواخت روی rho در (-1,1)
def corr_bayes(r, n, rho_min=0.5):
    z = np.arctanh(np.clip(r, -0.999, 0.999)); se = 1 / np.sqrt(n - 3)
    rho = np.linspace(-0.999, 0.999, 8001); step = rho[1] - rho[0]
    ll = st.norm.logpdf(z, np.arctanh(rho), se); ll0 = st.norm.logpdf(z, 0, se)
    lw = ll + np.log(0.5) + np.log(step) - ll0
    bf10 = np.exp(logsumexp(lw)); bfp0 = np.exp(logsumexp(lw[rho > 0]) + np.log(2))
    p_gt = float(np.exp(logsumexp(ll[rho > rho_min]) - logsumexp(ll)))
    return bf10, bfp0, p_gt

inf = T2[T2.lens.isin(INFORMATIVE)]; models = sorted(inf.model.unique()); rows4 = []
for metric in ["CP_delta", "EA_delta"]:
    piv = inf.pivot_table(index=["task", "tier", "lens"], columns="model", values=metric)
    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            pr = piv[[models[i], models[j]]].dropna(); r = st.spearmanr(pr.iloc[:, 0], pr.iloc[:, 1]).statistic
            bf10, bfp0, p5 = corr_bayes(r, len(pr))
            rows4.append(dict(ID="H4", metric=metric, pair=f"{models[i][:12]} ~ {models[j][:12]}", n_cells=len(pr), spearman_rho=r, BF10=bf10, BF_plus0=bfp0, P_rho_gt_0_5=p5))
h4 = pd.DataFrame(rows4); h4.to_csv("bayes_H4_cross_encoder.csv", index=False); print(h4.round(3).to_string(index=False))

## خلاصه و ذخیره

In [ ]:
res["BF10_two_sided_fmt"] = res.BF10_two_sided.map(fmt); res["BF_dir_main_fmt"] = res.BF_dir_main.map(fmt); res["BF_dir_min_fmt"] = res.BF_dir_min.map(fmt)
res.to_csv("bayes_all_tests.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 80, "display.max_rows", 200)
show = res[["ID", "test", "mean", "sd", "BF_dir_main_fmt", "BF_dir_min_fmt", "category", "prior_sensitive"]].copy()
show["mean"] = show["mean"].round(2); show["sd"] = show["sd"].round(2)
print(show.to_string(index=False))

# خلاصه به تفکیک فرضیه
summ = []
for hid, g in res.groupby("ID", sort=False):
    vc = g.category.value_counts().to_dict()
    summ.append(dict(ID=hid, n_tests=len(g), **{k: vc.get(k, 0) for k in ["STRONG for effect", "MODERATE for effect", "INCONCLUSIVE", "MODERATE for null", "STRONG for null", "EVIDENCE FOR OPPOSITE DIRECTION"]}))
summ = pd.DataFrame(summ); summ.to_csv("bayes_summary_by_hypothesis.csv", index=False); print(); print(summ.to_string(index=False))

notes = ["# Bayesian follow-up (post hoc)", "",
 "Unit of analysis: independent resampling seeds (n=5), lens-averaged. JZS t-test, Cauchy prior r in {0.5, 0.707, 1.0}; category from the minimum directional BF over r.",
 "H5 (lens invariance) and H6 (parameter grid) are NOT analysed here: their units (lens variants, grid configurations) are not independent, so BFs would be inflated.",
 "H1/H2 use the primary encoder only (the other encoders have a single seed). H4 is descriptive (cells share tasks/tiers).",
 "These results do not change the pre-specified verdicts; they quantify strength of evidence, including evidence for the null."]
open("BAYES_NOTES.md", "w").write("\n".join(notes))
import shutil; os.makedirs("bayes_out", exist_ok=True)
for f in ["bayes_all_tests.csv", "bayes_summary_by_hypothesis.csv", "bayes_H4_cross_encoder.csv", "BAYES_NOTES.md"]: shutil.copy(f, "bayes_out")
shutil.make_archive("bayes_results", "zip", "bayes_out")
try:
    from google.colab import files; files.download("bayes_results.zip")
except Exception: print("bayes_results.zip ساخته شد")